# 第 7 周练习：用 QLoRA 微调开源模型做价格预测

## 练习目标

对 Hugging Face 数据集 `ed-donner/pricer-data` **抽一小部分子集**，用 **QLoRA** 微调 `Llama-3.2-1B`，再在测试子集上算 **MAE / MSE / R²**，并画预测 vs 实际散点图。

本笔记本设计为在带 GPU（T4 或 A100）的 **Google Colab** 中运行：完整数据从 Hub 加载，本地不必自备副本。

## 和本课第 7 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| QLoRA / 4-bit | `BitsAndBytesConfig` + `prepare_model_for_kbit_training` |
| prompt / completion SFT | `format_for_training` + `completion_only_loss=True` |
| PEFT LoRA | `LoraConfig` / `PeftModel` |
| 价格抽取与评估 | 正则 + `mean_squared_error` / `r2_score` |

## 要求

- Colab Secrets 里配置 `HF_TOKEN`
- 在 Hugging Face 上接受 [Meta Llama 3.2 许可证](https://huggingface.co/meta-llama/Llama-3.2-1B)
- 可选：`WANDB_API_KEY`（仅当 `LOG_TO_WANDB=True`）


In [ ]:
# ========== 依赖安装：Colab 里跑一次即可 ==========

# -q 安静安装 QLoRA / SFT 常用栈
%pip install -q datasets transformers torch peft bitsandbytes trl accelerate
# 若后面把 LOG_TO_WANDB 设为 True，再取消下一行注释安装 wandb
# %pip install -q wandb


In [ ]:
# ========== 导入：数据、量化模型、训练、评估、画图 ==========

# 标准库 os：写环境变量（如 WANDB_API_KEY）
import os
# 标准库 re：从模型输出里正则抽价格
import re
# 标准库 math：数值辅助（本笔记本可能间接用到）
import math
# tqdm：评估循环进度条
from tqdm import tqdm
# Colab Secrets：读 HF_TOKEN / WANDB_API_KEY
from google.colab import userdata
# Hugging Face Hub 登录
from huggingface_hub import login
# PyTorch
import torch
# Transformers：因果 LM、分词器、量化配置、固定种子
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
# 从 Hub 加载数据集
from datasets import load_dataset
# PEFT：LoRA 配置、加载适配器、k-bit 训练准备
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
# TRL：SFT Trainer / Config
from trl import SFTTrainer, SFTConfig
# 画散点图
import matplotlib.pyplot as plt
# sklearn：MSE 与 R²
from sklearn.metrics import mean_squared_error, r2_score


## 配置说明

按你的 GPU / 账号改下一格常量：

- **HF_USER：** 你的 Hugging Face 用户名（推送模型用）
- **TRAIN_SUBSET_SIZE：** T4 建议 2k–5k；A100 可提到 10k–20k
- **BATCH_SIZE：** T4 约 2–4；A100 约 8–16


In [ ]:
# ========== 超参数与路径：集中配置，后面只引用常量 ==========

# 底座模型 id（字符串必须保持原样）
BASE_MODEL = "meta-llama/Llama-3.2-1B"
# 商品描述+价格数据集
DATASET_NAME = "ed-donner/pricer-data"
# Hub 用户名；注释提醒改成你自己的
HF_USER = "mideseniordev"  # Change to your HuggingFace username

# ----- 子集规模：先拉全量再采样 -----
TRAIN_SUBSET_SIZE = 5000
TEST_SUBSET_SIZE = 500
# 从训练子集再划出验证比例
VAL_RATIO = 0.1
# 打乱种子，保证可复现
RANDOM_SEED = 42

# ----- 序列与 prompt 模板（模板字符串必须保持英文原样） -----
MAX_SEQUENCE_LENGTH = 182
QUESTION = "What does this cost to the nearest dollar?"
PREFIX = "Price is $"

# ----- QLoRA -----
LORA_R = 16
LORA_ALPHA = 32
# 挂 LoRA 的注意力投影模块名
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1

# ----- 训练 -----
EPOCHS = 1
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"
SAVE_STEPS = 500
# 默认关 W&B；有密钥时再改 True
LOG_TO_WANDB = False  # Set True if you have WANDB_API_KEY in Colab secrets

# Jupyter 内联显示 matplotlib 图
%matplotlib inline


## 登录 Hugging Face

在 Colab：**Secrets**（钥匙图标）→ 新建秘密 → 名称填 `HF_TOKEN`，值贴你的 Access Token。


In [ ]:
# ========== 用 Colab Secret 登录 Hub ==========

# 读取名为 HF_TOKEN 的秘密（名称必须保持原样）
hf_token = userdata.get("HF_TOKEN")
# 登录；写入 git credential 便于后续 push
login(hf_token, add_to_git_credential=True)


## 加载数据集并抽取子集

从 Hub 加载 `ed-donner/pricer-data`，打乱后截取训练/测试子集，再从训练子集划出验证集。无需本地数据副本。


In [ ]:
# ========== 加载全量数据 → 采样子集 → train/val 划分 ==========

# 按 DATASET_NAME 从 Hub 拉取 DatasetDict
dataset = load_dataset(DATASET_NAME)
# 原始 train / test split
train_full = dataset["train"]
test_full = dataset["test"]

# 固定种子 shuffle 后 select，保证可复现且多样
train_subset = train_full.shuffle(seed=RANDOM_SEED).select(
    range(min(TRAIN_SUBSET_SIZE, len(train_full)))
)
test_subset = test_full.shuffle(seed=RANDOM_SEED).select(
    range(min(TEST_SUBSET_SIZE, len(test_full)))
)

# 再把 train_subset 按 VAL_RATIO 拆成 train / val（这里的 test key 实际是验证集）
split = train_subset.train_test_split(test_size=VAL_RATIO, seed=RANDOM_SEED)
train_data = split["train"]
val_data = split["test"]

# 打印三份规模，确认截断生效
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_subset):,}")


In [ ]:
# ========== 格式化为 SFT 的 prompt / completion（供 completion_only_loss） ==========

# 定价数据字段：text=描述，price=真值价格
def format_for_training(example):
    # 商品描述
    desc = example["text"]
    # 转 float，后面 round 成整数美元
    price = float(example["price"])
    # 拼提问模板 + 描述 + 价格前缀（QUESTION/PREFIX 字符串勿改）
    prompt = f"{QUESTION}\n\n{desc}\n\n{PREFIX}"
    # completion 只含价格数字，便于只在价格 token 上算 loss
    completion = f"{int(round(price))}.00"
    return {"prompt": prompt, "completion": completion, "price": price}

# map 到 train / val
train_formatted = train_data.map(format_for_training)
val_formatted = val_data.map(format_for_training)

# 预览一条，确认格式
ex = train_formatted[0]
print("Example formatted example:")
print(f"Prompt: {ex['prompt'][:120]}...")
print(f"Completion: {ex['completion']}")


## 加载模型与分词器

使用 **4-bit 量化（QLoRA）** 加载底座，显著降低显存占用，便于在 T4 上微调。


In [ ]:
# ========== 4-bit 底座 + tokenizer + k-bit 训练准备 ==========

# NF4 + double quant；计算 dtype 用 bfloat16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

# 加载与底座匹配的分词器
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# pad 用 eos；右侧 padding 适合因果 LM
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 量化加载因果 LM；device_map=auto 自动放 GPU
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成时 pad_token_id 对齐
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
# 为 k-bit 训练做梯度检查点等准备
base_model = prepare_model_for_kbit_training(base_model)

# 打印显存占用（MB）
print(f"Model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")


## QLoRA + 训练设置

- **prompt + completion + `completion_only_loss=True`：** 损失主要落在价格 token 上
- **`LoraConfig`：** 在注意力投影层挂低秩适配器
- **`SFTConfig`：** 监督微调的学习率、精度、评估与保存策略


In [ ]:
# ========== 探测 GPU 精度能力，并组装 LoRA + SFTConfig ==========

# datetime：给本次 run 起带时间戳的名字
from datetime import datetime

# 有 CUDA 才算 GPU；bf16 需要 Ampere+（算力主版本 >= 8）
use_gpu = torch.cuda.is_available()
use_bf16 = use_gpu and torch.cuda.get_device_capability()[0] >= 8
# 没 GPU 时打印英文警告（文案必须保持原样，便于 Colab 用户检索）
if not use_gpu:
    print("WARNING: No GPU detected. Enable GPU in Colab: Runtime → Change runtime type → GPU")

# 本次输出目录与 Hub 仓库名
RUN_NAME = f"pricer-subset-{datetime.now():%Y-%m-%d_%H.%M.%S}"
OUTPUT_DIR = f"./{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{RUN_NAME}"

# LoRA：挂到 TARGET_MODULES
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# SFT 训练参数；max_length 在新版 trl 替代旧名 max_seq_length
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,  # was max_seq_length in older trl
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIMIZER,
    weight_decay=0.001,
    max_grad_norm=0.3,
    # 无 GPU 时强制 CPU 路径
    use_cpu=not use_gpu,
    # GPU 且非 bf16 → fp16；Ampere+ → bf16
    fp16=use_gpu and not use_bf16,
    bf16=use_bf16,
    logging_steps=25,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=250,
    # 开 W&B 才 report 到 wandb，否则 "none"
    report_to="wandb" if LOG_TO_WANDB else "none",
    # 只在 completion token 上算 loss
    completion_only_loss=True,  # train only on completion tokens
)


In [ ]:
# ========== 可选：登录 Weights & Biases ==========

# 仅当 LOG_TO_WANDB=True 时才 import / login，避免无依赖环境报错
if LOG_TO_WANDB:
    import wandb
    # 从 Colab Secrets 读密钥名 WANDB_API_KEY
    wandb_api_key = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login()


## 训练

创建 `SFTTrainer`，启动 `train()`，并把适配器与分词器保存到 `OUTPUT_DIR`。


In [ ]:
# ========== 创建 Trainer → 训练 → 本地保存 ==========

# 注入底座、SFT 参数、格式化后的 train/val、LoRA 配置
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    peft_config=lora_config,
)

# 真正开始微调（耗时取决于子集大小与 GPU）
trainer.train()
# 保存适配器权重等到输出目录
trainer.save_model(OUTPUT_DIR)
# 同步保存 tokenizer，评估时好加载
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")


## 评估

重新加载底座 + 本地适配器，在测试子集上预测价格并计算指标。


In [ ]:
# ========== 评估用：底座 + 本地 PEFT 适配器 ==========

# 再按同一量化配置加载底座（评估阶段独立加载）
base_for_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 从 OUTPUT_DIR 挂上刚训好的适配器
fine_tuned_model = PeftModel.from_pretrained(base_for_eval, OUTPUT_DIR)
# 用保存目录里的 tokenizer，保证词表一致
tokenizer_eval = AutoTokenizer.from_pretrained(OUTPUT_DIR)
# 评估模式：关掉 dropout 等训练行为
fine_tuned_model.eval()


In [ ]:
# ========== 价格抽取与单条预测函数 ==========

def extract_price(response: str) -> float:
    """Extract numeric price from model output."""
    # 若输出里含 PREFIX，只取其后半段（更像「续写的价格」）
    if PREFIX in response:
        tail = response.split(PREFIX)[-1]
    else:
        tail = response
    # 去掉千分位逗号
    tail = tail.replace(",", "")
    # 抓第一个数字（可带符号/小数点）
    match = re.search(r"[-+]?\d*\.?\d+", tail)
    return float(match.group()) if match else 0.0


def predict_price(datapoint, model, tokenizer) -> float:
    """Predict price for a single datapoint."""
    # 固定种子，降低生成随机性带来的抖动
    set_seed(42)
    # 模型当前所在设备（GPU/CPU）
    device = next(model.parameters()).device
    # 拼与训练一致的提问 + 描述 + 价格前缀
    prompt = f"{QUESTION}\n\n{datapoint['text']}\n\n{PREFIX}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # 推理不计梯度
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    # 整段 decode 后再用 extract_price 抽数字
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_price(response)


In [ ]:
# ========== 在测试子集上批量评估并打印 MAE / MSE / R² ==========

# 最多评估 200 条，避免太慢
EVAL_SIZE = min(200, len(test_subset))
predictions = []
ground_truths = []

# 逐条预测；desc 字符串给进度条标题（保持英文原样）
for i in tqdm(range(EVAL_SIZE), desc="Evaluating"):
    dp = test_subset[i]
    pred = predict_price(dp, fine_tuned_model, tokenizer_eval)
    truth = float(dp["price"])
    predictions.append(pred)
    ground_truths.append(truth)

# 平均绝对误差（手写 MAE）
mae = sum(abs(p - t) for p, t in zip(predictions, ground_truths)) / len(predictions)
# 均方误差
mse = mean_squared_error(ground_truths, predictions)
# R² 转成百分比展示
r2 = r2_score(ground_truths, predictions) * 100

print(f"MAE: ${mae:,.2f}")
print(f"MSE: {mse:,.0f}")
print(f"R²: {r2:.1f}%")


In [ ]:
# ========== 散点图：预测价格 vs 实际价格 ==========

# 正方形画布，方便对照 y=x
plt.figure(figsize=(8, 8))
# 坐标轴上限：两边最大值，至少为 1 避免空图
max_val = max(max(ground_truths), max(predictions), 1)
# 半透明散点
plt.scatter(ground_truths, predictions, alpha=0.5)
# 理想线：完美预测时点落在这条对角线上（label 英文保持原样）
plt.plot([0, max_val], [0, max_val], "k--", label="Perfect prediction")
plt.xlabel("Actual price ($)")
plt.ylabel("Predicted price ($)")
plt.title(f"Fine-tuned Model: MAE=${mae:,.2f}")
plt.legend()
plt.tight_layout()
plt.show()


## 预测示例

抽查前几条测试样本，肉眼看描述、预测价与真值误差。


In [ ]:
# ========== 打印前 5 条样例：描述 / 预测 / 实际 / 误差 ==========

for i in range(5):
    dp = test_subset[i]
    pred = predict_price(dp, fine_tuned_model, tokenizer_eval)
    truth = float(dp["price"])
    # 描述过长则截断并加省略号，便于阅读
    desc = dp["text"][:60] + "..." if len(dp["text"]) > 60 else dp["text"]
    print(f"Desc: {desc}")
    print(f"  Predicted: ${pred:,.2f} | Actual: ${truth:,.2f} | Error: ${abs(pred - truth):,.2f}")
    print()


## 推送到 Hugging Face Hub

把本次微调产出上传到 `HUB_MODEL_NAME`，方便以后 `PeftModel.from_pretrained` 复用。


In [ ]:
# ========== 上传：Trainer 一键推，或 HfApi 传整个文件夹 ==========

# 用 Trainer 推到 Hub（仓库 id 为 HUB_MODEL_NAME）
trainer.push_to_hub(HUB_MODEL_NAME)
# 备选：手动上传 OUTPUT_DIR 里的适配器与配置
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HUB_MODEL_NAME, repo_type="model")
